# Pancake Engine 0.3 — walk-forward tour

**Engine 0.3 is correctness-first, not TS parity. Known TS divergences are documented in `pancake-production/docs/research/pancake-engine-0.3-ts-divergences.md`.**

This notebook is a rendering of test-pinned facts. It contains **no novel logic** — every value below is also asserted in `tests/`. The notebook is a tour, not a spec.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pancake_engine').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pancake_engine import (
    BacktestConfig, WalkforwardConfig,
    load_dataset, load_spec,
    run_backtest, run_walkforward,
)
print(f'pancake_engine OK; project root = {PROJECT_ROOT}')

## Example 1 — toy (synthetic 50-row regression sentinel)

In [ ]:
ex = PROJECT_ROOT / 'examples' / 'toy'
spec = load_spec(ex / 'spec.json')
dataset = load_dataset(ex / 'dataset.json')
DAY = 86_400
result = run_backtest(spec, dataset, BacktestConfig(observation_time=50 * DAY))
print(f'result_hash    = {result.result_hash}')
print(f'num_trades     = {result.metrics.standard.num_trades}')
print(f'total_return   = {result.metrics.standard.total_return:.4f}')
print(f'win_rate       = {result.metrics.standard.win_rate}')
print(f'max_drawdown   = {result.metrics.standard.max_drawdown:.4f}')
print(f'warnings       = {[w.code.value for w in result.warnings]}')

## Example 2 — Jakarta temperature (synthetic, hot-day market)

In [ ]:
ex = PROJECT_ROOT / 'examples' / 'jakarta_temperature'
spec = load_spec(ex / 'spec.json')
dataset = load_dataset(ex / 'dataset.json')
import json
expected = json.loads((ex / 'expected_result.json').read_text())
result = run_backtest(spec, dataset, BacktestConfig(observation_time=expected['observation_time']))
print(f'result_hash    = {result.result_hash}')
print(f'num_trades     = {result.metrics.standard.num_trades}')
print(f'total_return   = {result.metrics.standard.total_return:.4f}')

## Example 3 — rapture family (synthetic, NO-side extreme prices)

In [ ]:
ex = PROJECT_ROOT / 'examples' / 'rapture_family'
spec = load_spec(ex / 'spec.json')
dataset = load_dataset(ex / 'dataset.json')
expected = json.loads((ex / 'expected_result.json').read_text())
result = run_backtest(spec, dataset, BacktestConfig(observation_time=expected['observation_time']))
print(f'result_hash    = {result.result_hash}')
print(f'num_trades     = {result.metrics.standard.num_trades}')
print(f'total_return   = {result.metrics.standard.total_return:.4f}')
print(f'win_rate       = {result.metrics.standard.win_rate}')

## Example 4 — BTC prediction-market hedge (synthetic) via walk-forward

This is the only example that runs through the PR-2 walk-forward layer.
Aggregate result hash matches the committed `expected_result.json`.

In [ ]:
ex = PROJECT_ROOT / 'examples' / 'btc_pred_hedge'
spec = load_spec(ex / 'spec.json')
dataset = load_dataset(ex / 'dataset.json')
expected = json.loads((ex / 'expected_result.json').read_text())
wf_config = WalkforwardConfig(window_type='anchored', test_horizon='2QS', step='2QS', min_fold_count=2)
bt_config = BacktestConfig(observation_time=expected['observation_time'])
result = run_walkforward(spec, dataset, wf_config, bt_config)
print(f'aggregate_result_hash = {result.aggregate_result_hash}')
print(f'fold_count            = {result.aggregate.fold_count}')
print(f'pooled_num_trades     = {result.aggregate.pooled.num_trades}')
print(f'fold_sharpe_dispersion = {result.aggregate.fold_sharpe_dispersion}')
print(f'warnings              = {[w.code.value for w in result.warnings]}')

## Doctrine reminder

Engine 0.3 is **correctness-first, not TS parity**. Known TS divergences are documented in `pancake-production/docs/research/pancake-engine-0.3-ts-divergences.md`:

- D-1 cash leak (Engine 0.3 prevents)
- D-2 `Date.now()` non-determinism (Engine 0.3 refuses)
- D-3 equity-curve sampling (Engine 0.3 consistent)
- D-4 / D-5 silent skips (Engine 0.3 emits warnings)
- D-6 `brier_*` always null (Engine 0.3 computes `brier_crowd`)
- D-11 fee realized at entry (Engine 0.3 marks at `shares × entry_fill_price`)
- D-12 negative `slippage_bps` / `fee_bps` rejected
- D-13 Sortino denominator (Engine 0.3 uses full sample size)
- D-14 daily resampling (Engine 0.3 carries forward)

If receipts disagree with the TS evidence-runner, the divergence is documented and intentional.